# Lesson 09 — Routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/09-routing-and-multi-index/colab.ipynb)

*When one component is better for an identifiable slice of traffic, classify the traffic instead of compromising globally.*

**What you will learn**

- Why routing is the principled fix for lesson 04's failure (and weighted RRF is not)
- A heuristic router: five regexes that recognise identifier-shaped queries
- Asymmetric error costs, and how they should set your defaults
- A router that was *worse* than no router — and the design flaw behind it
- Heuristic vs LLM routing, and the general routing pattern

| | |
|---|---|
| ⏱️ Time | 40 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [08-parent-doc-and-metadata](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/08-parent-doc-and-metadata/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Unfinished business from lesson 04

Hybrid search lost to dense retrieval at the top of the ranking, for a specific,
measurable reason: RRF weights every retriever equally, and on vocabulary-mismatch
questions BM25 is *confidently wrong*. Two candidate fixes:

- **Weighted RRF** — give dense more votes. It would work, and it is tuning a global
  constant against 44 questions: overfitting with extra steps, applying one compromise
  to every query regardless of what the query needs.
- **Routing** — decide *per query*. This lesson.

## 2. The insight: which retriever wins is predictable

| Query | Wins | Why |
|---|---|---|
| "What does exit status 75 mean?" | BM25 | the literal token *is* the query |
| "What is `FUSION_CONSTANT` set to?" | BM25 | exact key name, no semantic content |
| "Why do I get charged twice?" | dense | shares no vocabulary with the answer |
| "My scanned paperwork comes back blank" | dense | needs meaning, not matching |

Identifiers, config keys, error codes and version numbers are *typographically marked*.
A regex recognises them — and a regex costs nothing and can be read.

In [ ]:
from ragkit.retrieve.router import IDENTIFIER_PATTERNS, looks_lexical

for pattern in IDENTIFIER_PATTERNS:
    print(pattern.pattern)

In [ ]:
import pandas as pd


def which_pattern(query):
    return next((p.pattern for p in IDENTIFIER_PATTERNS if p.search(query)), "")


probes = [
    "What does exit status 75 mean?",
    "What is FUSION_CONSTANT set to?",
    "Error TX-4491 when indexing",
    "Is this fixed in v2.14.3?",
    "What does reciprocal_rank_fusion(rankings, k) return?",
    "Why do I get charged twice?",
    "How do I make search results more relevant?",
    "What changed in Python 3.12?",  # a version number in a prose question
]
pd.DataFrame(
    [
        {
            "query": q,
            "route": "lexical" if looks_lexical(q) else "semantic",
            "pattern that fired": which_pattern(q),
        }
        for q in probes
    ]
)

🔍 The last probe is a prose question that happens to contain a version number, and it
gets routed lexically. Is that right? That depends on the corpus — and it is exactly
the kind of decision you can *see and reason about* with a regex, and cannot with a
model that returns a label and no reason.

## 3. The asymmetry that sets the default

The router is deliberately **conservative about choosing lexical**, because the two
possible mistakes do not cost the same:

- **Prose question sent to BM25** → bad. BM25 cannot bridge vocabulary mismatch at all;
  the right chunk may not appear at *any* depth.
- **Identifier question sent to dense** → survivable. Dense retrieval on identifiers is
  weak but not blind, and the surrounding prose often carries it.

Let's check that claim on the corpus:

In [ ]:
from ragkit.pipeline import RagPipeline
from ragkit.retrieve import Bm25Retriever, DenseRetriever

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())
dense = DenseRetriever(pipeline._store, pipeline.embedder)
lexical = Bm25Retriever(pipeline._bm25)


def doc_rank(retriever, question, relevant, depth=50):
    docs = list(dict.fromkeys(s.chunk.doc_id for s in retriever.retrieve(question, top_k=depth)))
    return next((i for i, d in enumerate(docs, 1) if d in relevant), "absent")


from ragkit.eval.golden import load_golden

rows = []
for q in load_golden():
    if q.kind in ("vocab_mismatch", "lookup"):
        rows.append(
            {
                "kind": q.kind,
                "question": q.question[:58],
                "doc rank via dense": doc_rank(dense, q.question, q.relevant),
                "doc rank via BM25": doc_rank(lexical, q.question, q.relevant),
            }
        )
ranks = pd.DataFrame(rows)
display(ranks)
numeric = ranks.replace("absent", 99)
numeric[["doc rank via dense", "doc rank via BM25"]] = numeric[
    ["doc rank via dense", "doc rank via BM25"]
].astype(int)
numeric.groupby("kind")[["doc rank via dense", "doc rank via BM25"]].agg(["mean", "max"])

Read it as two "wrong route" scenarios:

- **Lookup questions sent to dense** (the identifier mistake): look at the *dense*
  column for `lookup`. Weak but not blind — the right document is still at or near the
  top.
- **Vocabulary-mismatch questions sent to BM25** (the prose mistake): look at the *BM25*
  column for `vocab_mismatch`, especially the **worst case**. It is not uniformly worse
  — BM25 occasionally gets lucky on a shared word — but its misses go deeper, because
  it has no mechanism at all for bridging vocabulary.

Defaults are set by the worst case, not the average.

**Asymmetric costs should decide your defaults.** When the router is unsure, it chooses
the failure that degrades gracefully. `LlmRouter` falls back to semantic on any error or
unexpected reply for the same reason. This reasoning generalises far beyond routing.

## 4. What it routes on the golden set

In [ ]:
!{sys.executable} lessons/09-routing-and-multi-index/run.py | tail -14

Two things to read off the summary:

- **Zero `vocab_mismatch` questions go to lexical.** Those are precisely the questions
  BM25 damages, and the router protects all of them — lesson 04's failure fixed at its
  root rather than dampened by a tuned constant.
- **Only a handful of questions are diverted at all**, so the aggregate effect will be
  small *by construction*. Judge a router on whether the questions it moved were the
  right ones, not on the headline number.

## 5. Measure it — including the version that was worse

The first implementation routed lexical queries to **bare BM25** and everything else to
the full reranked path. Let's rebuild that version next to the current one (where
**both** branches are reranked).

In [ ]:
from ragkit.eval.run import run_eval
from ragkit.retrieve import HeuristicRouter, HybridRetriever
from ragkit.retrieve.rerank import RerankingRetriever

n = pipeline.settings.rerank_candidates
semantic = RerankingRetriever(
    HybridRetriever([dense, lexical], candidates=n), pipeline.reranker, candidates=n
)

runs = {"rerank (no routing)": run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)}

pipeline._retriever = HeuristicRouter(lexical, semantic)  # the first version
runs["router, bare BM25 branch"] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)

pipeline.set_strategy("router")  # both reranked
runs["router, both branches reranked"] = run_eval(
    pipeline=pipeline, retrieval_only=True, quiet=True
)

display(compare_runs(runs).round(3))
print("lookup questions only:")
display(compare_runs(runs, kind="lookup").round(3))

### What the repo measured

| Pipeline | lookup R@1 | ALL R@1 | nDCG@5 |
|---|---|---|---|
| rerank (no routing) | 1.000 | 0.811 | 0.946 |
| router, bare BM25 branch | **0.900** | 0.818 | 0.946 |
| **router, both branches reranked** | **1.000** | **0.841** | **0.955** |

The bare-BM25 router lost a lookup question: for "What does exit status 75 from the
indexer mean?", BM25 alone put `09-query-understanding.md` first instead of
`07-operations.md`. The reranker would have fixed it — the lexical branch simply did not
have one.

**The design flaw: the router was deciding two things at once.** Its job is choosing
the candidate *source*. Whether to rerank is a separate decision, and reranking helps
whichever source was picked. Conflating them threw away lesson 05's gains on every
routed query.

> **When you add a component that selects between paths, check that every path still
> has everything the unrouted version had.** It is easy to build a router that quietly
> downgrades the branch it sends traffic to.

With the current defaults the differences are one or two questions either way — check
your table above, and remember how few questions are routed at all.

## 6. Heuristic or model?

`LlmRouter` asks the generator to classify each query. More flexible, and it costs a
full generation before retrieval starts. Compare its decisions with the regex:

In [ ]:
from ragkit.eval.golden import load_golden
from ragkit.retrieve.router import ROUTER_SYSTEM

if LLM_READY:
    rows = []
    for q in load_golden():
        reply, _ = pipeline.generator.generate(
            system=ROUTER_SYSTEM, prompt=f"Query: {q.question}\n\nClassification:", max_tokens=8
        )
        rows.append(
            {
                "kind": q.kind,
                "question": q.question[:60],
                "regex": "lexical" if looks_lexical(q.question) else "semantic",
                "LLM": "lexical" if "LEXICAL" in reply.upper() else "semantic",
            }
        )
    decisions = pd.DataFrame(rows)
    print(f"agreement: {(decisions.regex == decisions.LLM).mean():.0%}")
    display(pd.crosstab(decisions.kind, [decisions.regex, decisions.LLM]))
    display(decisions[decisions.regex != decisions.LLM])

Wherever the model routes a `vocab_mismatch` question lexically, it has made the
*expensive* mistake from section 3 — and it paid a model call to do it. On this corpus
the regex is as good and free, which is the usual outcome when the thing being
classified is a *surface property of the text*.

Reach for a model when the decision needs world knowledge a regex cannot express — "is
this about our product or the industry?", "does this need live data?" — not for "does
this contain an identifier?".

## 7. The general pattern

**When one component is systematically better for an identifiable subset of traffic,
classify the traffic instead of compromising globally.** The same shape:

- cheap model for simple questions, expensive model for hard ones
- skip reranking when retrieval is already confident
- skip query transformation on keyword-shaped queries (lesson 06's exercise)
- send time-sensitive questions to a live source rather than the index

Every one is the same decision: a global compromise, or a per-query choice.

## 8. Check your understanding

**1. Why is the router conservative about routing to BM25?**

<details><summary>▶ Show answer</summary>

Because the mistakes are asymmetric: BM25 on a prose question can miss the answer
entirely (no vocabulary overlap), while dense on an identifier question is merely
weaker. When unsure, take the mistake that degrades gracefully.

</details>

**2. The first router was worse than no router. What was the design flaw?**

<details><summary>▶ Show answer</summary>

It coupled two decisions — which candidate source, and whether to rerank — so
routing to BM25 also silently dropped the reranker. Every path must keep everything
the unrouted pipeline had.

</details>

**3. Name one advantage of a regex router beyond cost.**

<details><summary>▶ Show answer</summary>

Debuggability: when a query is routed badly, you can see exactly which pattern
fired and fix it. A model returns a label with no reason.

</details>

## ✍️ Exercises

1. **Break it deliberately.** Add an over-eager pattern (e.g. any capitalised word) to a
   copy of `IDENTIFIER_PATTERNS` and re-run section 5. Which kind suffers?
2. **Route something else.** Build a router that skips reranking when the top dense
   score is above some threshold. How much latency does it save, and what does it cost
   in recall?
3. **When routing goes wrong:** find a realistic query that contains an identifier but
   is really a *conceptual* question.

## Key takeaways

1. Which retriever wins is predictable from the query's shape — so decide per query.
2. A few readable regexes route identifier-shaped queries for free.
3. Asymmetric error costs should set defaults: when unsure, choose semantic.
4. A router must not silently downgrade the branch it routes to — rerank both.
5. Use a model to route only when the decision needs knowledge a pattern cannot express.

## Next

**[Lesson 10 — Agentic RAG](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/10-agentic-rag/colab.ipynb)**